# CFA Youth Soccer Uniform Ordering Optimization
## MGMT 590-037: AI-Enhanced Optimization — Final Project
### Team 7 | Summer 2026

**Daniel Affleck · Satya Sai Usha Sree Ganteda · Aditya Ryan Gupta · Fabian Alexis Macias Peñalosa · Kentaro Nanayama · Siddharth Shenoy**

---

This notebook reproduces all results from the final report. It follows the full agent pipeline:

1. **Setup & Configuration** — Load the problem config with corrected $40,000 budget
2. **Data Loading & Exploration** — Load and inspect the cleaned order data
3. **Demand Prediction** — Train and compare 6 model variants (XGBoost, LightGBM, Ridge ± PTO)
4. **Model A: Single-Period Optimization** — Run 5 optimizers on the independent newsvendor
5. **Model B: Joint Two-Period Optimization** — Carryover inventory formulation (stakeholder modification)
6. **Baseline Comparison** — Agent vs. historical-average ordering
7. **Sensitivity Analysis** — Cost sensitivity to ±30% parameter shifts
8. **Summary & Recommendations**

### How to Run

```
uv run jupyter notebook report/final_notebook.ipynb
```

All paths are relative to the project root. Run cells in order.

---
## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
np.random.seed(42)

# Ensure project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) == "report":
    os.chdir(PROJECT_ROOT)
elif not os.path.exists("src"):
    PROJECT_ROOT = os.getcwd()

sys.path.insert(0, PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"Python version: {sys.version}")

# Style
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

In [ ]:
from src.models.problem_config import ProblemConfig, get_budget, get_moq, get_cost
from src.models.data_module import DataModule
from src.models.prediction_module import PredictionModule, PredictionParams
from src.models.all_optimizers_combined_final import compare_optimizers, optimize_with_slsqp
from src.models.joint_optimizer import optimize_joint_slsqp, joint_saa_exact, _compute_shadow_price_model_a

# Load and patch config: add 'year' to groupby_keys, set budget to $40,000
with open("data/default_config.json") as fh:
    raw_config = json.load(fh)

gk = raw_config["data_requirements"]["groupby_keys"]
if "year" not in gk:
    gk.insert(0, "year")
raw_config["data_requirements"]["test_years"] = [2025, 2026]
raw_config["data_requirements"]["train_years"] = [2020, 2021, 2022, 2023, 2024]

for c in raw_config["constraints"]:
    if "budget" in c.get("name", "").lower():
        for key in ["budget_usd", "budget_usd_per_season"]:
            if key in c.get("parameters", {}):
                c["parameters"][key] = 40000

config = ProblemConfig(**raw_config)

print(f"Problem: {config.problem_title}")
print(f"Budget:  ${get_budget(config):,.0f} per season")
print(f"MOQ:     {get_moq(config)} units")
print(f"Groupby: {config.data_requirements.groupby_keys}")
print(f"Train:   {config.data_requirements.train_years}")
print(f"Test:    {config.data_requirements.test_years}")

### Cost Structure

All costs expressed as fractions of selling price (`unit_price`).

In [ ]:
cost_data = []
for item in config.cost_structure:
    cost_data.append({
        "Name": item.name,
        "Value": item.value,
        "Unit": item.unit or "",
        "Category": item.product_category or "all",
        "Period": f"Year {item.period}" if item.period else "all",
        "Description": item.description or "",
    })
pd.DataFrame(cost_data)

---
## 2. Data Loading & Exploration

In [ ]:
# Load raw cleaned data for exploration
raw_df = pd.read_csv("data/cleaned_orders.csv", low_memory=False)
print(f"Raw dataset: {len(raw_df):,} rows × {len(raw_df.columns)} columns")
print(f"Columns: {list(raw_df.columns)}")
print(f"Null counts:\n{raw_df.isnull().sum()}")
print(f"\nTotal demand (units): {raw_df['quantity'].sum():,}")
raw_df.head(10)

In [ ]:
# Year-over-year demand
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

year_demand = raw_df.groupby("year")["quantity"].sum()
year_demand.plot(kind="bar", ax=axes[0], color="#2D6A9F", edgecolor="white")
axes[0].set_title("Total Demand by Year")
axes[0].set_ylabel("Units")
axes[0].set_xlabel("")
axes[0].bar_label(axes[0].containers[0], fmt="%,.0f", fontsize=9)

cat_year = raw_df.groupby(["year", "product_category"])["quantity"].sum().unstack(fill_value=0)
cat_year.plot(kind="bar", stacked=True, ax=axes[1], edgecolor="white",
              color=["#2D6A9F", "#CFB991", "#888888"])
axes[1].set_title("Demand by Year and Category")
axes[1].set_ylabel("Units")
axes[1].set_xlabel("")
axes[1].legend(title="Category")

plt.tight_layout()
plt.show()

print(f"\nOrders per year:\n{raw_df.groupby('year').size()}")
print(f"\nCategory breakdown:\n{raw_df['product_category'].value_counts()}")
print(f"\nGender/Age breakdown:\n{raw_df['gender_age'].value_counts()}")

In [ ]:
# Size distribution
fig, ax = plt.subplots(figsize=(12, 5))
size_demand = raw_df.groupby("size")["quantity"].sum().sort_values(ascending=False)
size_demand.plot(kind="bar", ax=ax, color="#2D6A9F", edgecolor="white")
ax.set_title("Total Demand by Size Code")
ax.set_ylabel("Units")
ax.set_xlabel("")
ax.bar_label(ax.containers[0], fmt="%,.0f", fontsize=8, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Load data through the pipeline's DataModule (aggregates to demand groups)
dm = DataModule(config)
bundle = dm.load("data/cleaned_orders.csv")

print(bundle.summary())
print(f"\nTrain years: {bundle.metadata['train_years']}")
print(f"Test years:  {bundle.metadata['test_years']}")
print(f"Features:    {bundle.metadata['feature_columns']}")
print(f"\nDemand pivot (first 10 rows):")
bundle.demand_pivot.head(10)

---
## 3. Demand Prediction

We compare six model variants on **newsvendor cost** (not RMSE) — the metric that matters for the downstream ordering decision.

The single-period critical ratio is:

$$CR = \frac{c_u}{c_u + c_o} = \frac{0.40}{0.40 + 0.70} \approx 0.364$$

This means the optimal policy orders to approximately the **36th percentile** of demand.

In [ ]:
# Run prediction module — trains all 6 models and selects winner by NV cost
params = PredictionParams(
    train_years=[2020, 2021, 2022, 2023, 2024],
    test_years=[2025, 2026],
)
pred = PredictionModule(config, params).run(bundle)
print(f"\n{pred.summary()}")

In [ ]:
# Visualize model comparison
models = {k: v for k, v in pred.all_costs.items() if k != "Baseline"}
baseline_cost = pred.all_costs["Baseline"]

fig, ax = plt.subplots(figsize=(10, 5))
names = list(models.keys())
costs = [models[n] for n in names]
colors = ["#2D6A9F" if n == pred.best_model_name else "#CFB991" for n in names]

bars = ax.barh(names, costs, color=colors, edgecolor="white", height=0.6)
ax.axvline(baseline_cost, color="red", linestyle="--", linewidth=1.5, label=f"Baseline (${baseline_cost:,.0f})")
ax.set_xlabel("Newsvendor Cost ($)")
ax.set_title("Prediction Model Comparison (lower is better)")
ax.legend()

for bar, cost in zip(bars, costs):
    pct = (cost / baseline_cost - 1) * 100
    ax.text(cost + 100, bar.get_y() + bar.get_height()/2,
            f"${cost:,.0f} ({pct:+.1f}%)", va="center", fontsize=9)

ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from the winning model
fig, ax = plt.subplots(figsize=(8, 4))
pred.feature_importance.plot(kind="barh", ax=ax, color="#2D6A9F", edgecolor="white")
ax.set_title(f"Feature Importance ({pred.best_model_name})")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 4. Model A: Single-Period Optimization (Independent Newsvendor)

The SAA (Sample Average Approximation) objective over demand scenarios:

$$\min_{q} \quad \frac{1}{N} \sum_{n=1}^{N} \sum_{s=1}^{S} \left[ c_o^s \cdot \max(q_s - D_{sn}, 0) + c_u^s \cdot \max(D_{sn} - q_s, 0) \right]$$

Subject to seasonal budget ≤ $40,000, MOQ ≥ 6, and non-negativity.

We compare **5 optimizers**: SLSQP, L-BFGS-B, Projected Gradient Descent, SGD, and Adam.

In [ ]:
# Run all 5 optimizers (this may take a few minutes due to SLSQP restarts)
comparison_df, all_results, best_a = compare_optimizers(pred, config)

print(f"\nBest optimizer: {best_a.method}")
print(f"Objective: ${best_a.objective_value:,.2f}")
print(f"Total spend: ${best_a.total_spend:,.2f}")
print(f"Feasible: {best_a.feasible}")

In [ ]:
# Optimizer comparison table
comparison_df

In [ ]:
# Visualize optimizer comparison
fig, ax = plt.subplots(figsize=(10, 4))
opt_names = comparison_df["optimizer"].values
opt_costs = comparison_df["objective_value"].values
colors = ["#2D6A9F" if n == best_a.method else "#CFB991" for n in opt_names]

bars = ax.barh(opt_names, opt_costs, color=colors, edgecolor="white", height=0.5)
ax.set_xlabel("Objective Value ($)")
ax.set_title("Model A — Optimizer Comparison")
for bar, cost in zip(bars, opt_costs):
    ax.text(cost + 200, bar.get_y() + bar.get_height()/2,
            f"${cost:,.0f}", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Best optimizer's order plan (first 15 rows)
print(f"Order plan: {len(best_a.order_plan)} SKUs")
best_a.order_plan.head(15)

---
## 5. Model B: Joint Two-Period Optimization (Stakeholder Modification)

The stakeholder modification recognizes that Year 1 surplus carries to Year 2 at **zero cost**, acting as a free hedge against the much higher Year 2 stockout cost (80% vs 40%).

$$\min_{q_1, q_2} \quad \frac{1}{N} \sum_n \sum_s \left[ c_{u1} (D_{1sn} - q_{1s})^+ + c_{o2} (q_{2s} + I_{sn} - D_{2sn})^+ + c_{u2} (D_{2sn} - q_{2s} - I_{sn})^+ \right]$$

where $I_{sn} = (q_{1s} - D_{1sn})^+$ is the carryover inventory.

In [ ]:
# Run joint two-period optimizer (Model B)
model_b = optimize_joint_slsqp(pred, config, n_restarts=10)
print(model_b.summary())

# Compute shadow prices
shadow_a = _compute_shadow_price_model_a(pred, config)

print(f"\n{'='*60}")
print(f"MODEL A vs MODEL B COMPARISON")
print(f"{'='*60}")
print(f"Model A total cost:       ${best_a.objective_value:>12,.2f}")
print(f"Model B total cost:       ${model_b.objective_value:>12,.2f}")

cost_of_myopia = best_a.objective_value - model_b.objective_value
reduction_pct = cost_of_myopia / best_a.objective_value * 100

print(f"Cost of myopia:           ${cost_of_myopia:>12,.2f}  ({reduction_pct:.1f}%)")
print(f"")
print(f"Model B Year 1 orders:    {model_b.q1.sum():>12,.0f} units")
print(f"Model B Year 2 orders:    {model_b.q2.sum():>12,.0f} units")
print(f"Expected carryover:       {model_b.expected_carryover.sum():>12,.1f} units")
print(f"Model B Y1 spend:         ${model_b.y1_spend:>12,.2f}")
print(f"Model B Y2 spend:         ${model_b.y2_spend:>12,.2f}")
print(f"Model B feasible:         {model_b.feasible}")
print(f"")
print(f"Shadow prices ($/$ extra Y1 budget):")
print(f"  Model A:                ${shadow_a:>12.4f}")
print(f"  Model B:                ${model_b.y1_shadow_price:>12.4f}")

In [ ]:
# Model A vs B cost comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost comparison
labels = ["Model A\n(Independent)", "Model B\n(Joint)"]
costs = [best_a.objective_value, model_b.objective_value]
colors = ["#CFB991", "#2D6A9F"]
bars = axes[0].bar(labels, costs, color=colors, edgecolor="white", width=0.5)
axes[0].set_ylabel("Total Two-Period Cost ($)")
axes[0].set_title("Cost of Myopia: Model A vs Model B")
axes[0].bar_label(bars, fmt="$%,.0f", fontsize=11)
axes[0].annotate(
    f"Savings: ${cost_of_myopia:,.0f}\n({reduction_pct:.1f}%)",
    xy=(0.5, min(costs) + (max(costs) - min(costs)) * 0.5),
    fontsize=12, ha="center", color="green", fontweight="bold",
)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Carryover visualization
key_cols = [c for c in ["season", "gender_age", "size"] if c in model_b.y1_df.columns]
top_carryover = pd.DataFrame({
    "sku": model_b.y1_df[key_cols].apply(lambda r: " / ".join(str(v) for v in r), axis=1).values,
    "carryover": model_b.expected_carryover,
}).nlargest(10, "carryover")

axes[1].barh(top_carryover["sku"], top_carryover["carryover"], color="#2D6A9F", edgecolor="white")
axes[1].set_xlabel("Expected Carryover (units)")
axes[1].set_title("Top 10 SKUs by Expected Carryover")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Stakeholder Questions (Plain Language)

1. **Should Year 1 orders increase or decrease?**
2. **How does Year 2 procurement change?**
3. **What is the cost of myopia?**
4. **Which model should guide decisions?**

In [ ]:
print("Q1: Should Year 1 orders increase or decrease?")
print(f"    → Year 1 orders: {model_b.q1.sum():.0f} units under Model B.")
print(f"      Year 1 surplus carries free to Year 2, hedging the costly")
print(f"      Year 2 stockout (80% of price vs 40% in Year 1).\n")

print("Q2: How does Year 2 procurement change?")
print(f"    → Year 2 orders: {model_b.q2.sum():.0f} units under Model B.")
print(f"      ~{model_b.expected_carryover.sum():.0f} carryover units from Year 1 supplement supply,")
print(f"      reducing the need to source a discontinued model.\n")

print("Q3: What is the cost of myopia?")
print(f"    → ${cost_of_myopia:,.2f} per two-year cycle ({reduction_pct:.1f}% higher cost)")
print(f"      when each year is optimized independently vs jointly.\n")

print("Q4: Which model should guide decisions?")
print(f"    → Model B (joint) for jerseys/tops — lifecycle + asymmetric costs.")
print(f"      Model A (independent) for bottoms/socks — no carryover benefit.")

---
## 6. Baseline Comparison

Compare the agent's newsvendor-optimal ordering policy against the **historical-average baseline** (proxy for the league president's current heuristic).

Policy: order to the empirical CR-quantile (0.3636) from training data 2020–2024, evaluated on 2025 actuals.

In [ ]:
# Baseline comparison: agent (CR-quantile) vs historical-average ordering
# Compute directly from per-(group, year) data

CR = 0.40 / (0.40 + 0.70)  # critical ratio
co_frac = 0.70
cu_frac = 0.40
MOQ = 6

# Aggregate demand by (year, season, product_category, gender_age, size)
agg = (
    raw_df.groupby(["year", "season", "product_category", "gender_age", "size"])
    .agg(demand=("quantity", "sum"), unit_price=("unit_price", "mean"))
    .reset_index()
)

train = agg[agg["year"].isin([2020, 2021, 2022, 2023, 2024])]
test_2025 = agg[agg["year"] == 2025].copy()

key_cols = ["season", "product_category", "gender_age", "size"]

# Agent policy: empirical CR-quantile from training data
agent_orders = (
    train.groupby(key_cols)["demand"]
    .quantile(CR)
    .reset_index()
    .rename(columns={"demand": "agent_qty"})
)
agent_orders["agent_qty"] = agent_orders["agent_qty"].apply(
    lambda x: max(int(np.ceil(x)), MOQ) if x > 0 else 0
)

# Baseline: historical mean
baseline_orders = (
    train.groupby(key_cols)["demand"]
    .mean()
    .reset_index()
    .rename(columns={"demand": "baseline_qty"})
)
baseline_orders["baseline_qty"] = baseline_orders["baseline_qty"].apply(
    lambda x: max(int(np.ceil(x)), MOQ) if x > 0 else 0
)

# Merge with 2025 actuals
eval_df = test_2025.merge(agent_orders, on=key_cols, how="left")
eval_df = eval_df.merge(baseline_orders, on=key_cols, how="left")
eval_df["agent_qty"] = eval_df["agent_qty"].fillna(0)
eval_df["baseline_qty"] = eval_df["baseline_qty"].fillna(0)

# Compute newsvendor costs
eval_df["co"] = co_frac * eval_df["unit_price"]
eval_df["cu"] = cu_frac * eval_df["unit_price"]

eval_df["agent_cost"] = (
    eval_df["co"] * np.maximum(eval_df["agent_qty"] - eval_df["demand"], 0) +
    eval_df["cu"] * np.maximum(eval_df["demand"] - eval_df["agent_qty"], 0)
)
eval_df["baseline_cost"] = (
    eval_df["co"] * np.maximum(eval_df["baseline_qty"] - eval_df["demand"], 0) +
    eval_df["cu"] * np.maximum(eval_df["demand"] - eval_df["baseline_qty"], 0)
)

agent_total = eval_df["agent_cost"].sum()
baseline_total = eval_df["baseline_cost"].sum()
savings = baseline_total - agent_total
savings_pct = savings / baseline_total * 100

print(f"{'='*50}")
print(f"BASELINE COMPARISON (2025 Backtest)")
print(f"{'='*50}")
print(f"Agent total NV cost:    ${agent_total:>10,.2f}")
print(f"Baseline total NV cost: ${baseline_total:>10,.2f}")
print(f"Savings:                ${savings:>10,.2f}  ({savings_pct:.1f}%)")
print(f"SKUs evaluated:         {len(eval_df)}")

# By category
print(f"\nBy category:")
for cat, grp in eval_df.groupby("product_category"):
    a = grp["agent_cost"].sum()
    b = grp["baseline_cost"].sum()
    s = b - a
    print(f"  {cat:<10}: agent ${a:>9,.2f} | baseline ${b:>9,.2f} | savings ${s:>9,.2f}")

In [ ]:
# Baseline comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total comparison
labels = ["Agent\n(Newsvendor)", "Baseline\n(Hist. Average)"]
totals = [agent_total, baseline_total]
bars = axes[0].bar(labels, totals, color=["#2D6A9F", "#CFB991"], edgecolor="white", width=0.5)
axes[0].set_ylabel("Total Newsvendor Cost ($)")
axes[0].set_title("Agent vs Baseline (2025 Backtest)")
axes[0].bar_label(bars, fmt="$%,.0f", fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[0].annotate(
    f"Savings: ${savings:,.0f} ({savings_pct:.1f}%)",
    xy=(0.5, agent_total + (baseline_total - agent_total) * 0.5),
    fontsize=12, ha="center", color="green", fontweight="bold",
)

# By category
cat_data = eval_df.groupby("product_category")[["agent_cost", "baseline_cost"]].sum()
x = np.arange(len(cat_data))
w = 0.35
axes[1].bar(x - w/2, cat_data["agent_cost"], w, label="Agent", color="#2D6A9F", edgecolor="white")
axes[1].bar(x + w/2, cat_data["baseline_cost"], w, label="Baseline", color="#CFB991", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cat_data.index)
axes[1].set_ylabel("Newsvendor Cost ($)")
axes[1].set_title("Cost by Category")
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

---
## 7. Sensitivity Analysis

How does total newsvendor cost change if the overage/underage cost fractions shift by ±30%?

In [ ]:
# Sensitivity analysis: hold the order plan fixed, shift co and cu by ±30%
shifts = [-30, -20, -10, 0, 10, 20, 30]
base_co = 0.70
base_cu = 0.40

co_results = []
cu_results = []

for shift in shifts:
    # Overage sensitivity
    co_mult = base_co * (1 + shift / 100)
    cost = (
        co_mult * eval_df["unit_price"] * np.maximum(eval_df["agent_qty"] - eval_df["demand"], 0) +
        base_cu * eval_df["unit_price"] * np.maximum(eval_df["demand"] - eval_df["agent_qty"], 0)
    ).sum()
    co_results.append({"shift": shift, "co": co_mult, "cost": cost, "pct_change": (cost / agent_total - 1) * 100})

    # Underage sensitivity
    cu_mult = base_cu * (1 + shift / 100)
    cost = (
        base_co * eval_df["unit_price"] * np.maximum(eval_df["agent_qty"] - eval_df["demand"], 0) +
        cu_mult * eval_df["unit_price"] * np.maximum(eval_df["demand"] - eval_df["agent_qty"], 0)
    ).sum()
    cu_results.append({"shift": shift, "cu": cu_mult, "cost": cost, "pct_change": (cost / agent_total - 1) * 100})

co_df = pd.DataFrame(co_results)
cu_df = pd.DataFrame(cu_results)

print("Overage Cost Sensitivity:")
print(co_df.to_string(index=False))
print(f"\nUnderage Cost Sensitivity:")
print(cu_df.to_string(index=False))

In [ ]:
# Sensitivity chart
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(co_df["shift"], co_df["pct_change"], "o-", color="#2D6A9F", linewidth=2,
        markersize=8, label="Overage cost ($c_o$)")
ax.plot(cu_df["shift"], cu_df["pct_change"], "s--", color="#CFB991", linewidth=2,
        markersize=8, label="Underage cost ($c_u$)")

ax.axhline(0, color="gray", linestyle=":", linewidth=0.8)
ax.axvline(0, color="gray", linestyle=":", linewidth=0.8)
ax.set_xlabel("Parameter Shift (%)")
ax.set_ylabel("Cost Change (%)")
ax.set_title("Sensitivity Analysis: Cost Response to ±30% Parameter Shifts")
ax.legend(fontsize=12)
ax.set_xlim(-35, 35)

# Add annotations at ±30%
ax.annotate(f"{co_df.iloc[-1]['pct_change']:+.1f}%", xy=(30, co_df.iloc[-1]["pct_change"]),
            xytext=(5, 10), textcoords="offset points", fontsize=10, color="#2D6A9F")
ax.annotate(f"{cu_df.iloc[-1]['pct_change']:+.1f}%", xy=(30, cu_df.iloc[-1]["pct_change"]),
            xytext=(5, -15), textcoords="offset points", fontsize=10, color="#CFB991")

plt.tight_layout()
plt.show()

co_range = co_df.iloc[-1]["pct_change"] - co_df.iloc[0]["pct_change"]
cu_range = cu_df.iloc[-1]["pct_change"] - cu_df.iloc[0]["pct_change"]
print(f"\nOverage cost sensitivity ratio: {co_range / cu_range:.1f}x more sensitive than underage cost")
print(f"The overage fraction (0.70) is the dominant cost driver to validate.")

---
## 8. Summary & Recommendations

In [ ]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

print(f"\n--- Data ---")
print(f"Dataset:          {len(raw_df):,} line items, 2020-2026")
print(f"Demand groups:    {bundle.metadata['demand_groups']}")
print(f"Train/Test split: {bundle.metadata['train_years']} / {bundle.metadata['test_years']}")

print(f"\n--- Prediction ---")
print(f"Best model:       {pred.best_model_name}")
print(f"NV cost:          ${pred.all_costs[pred.best_model_name]:,.0f}")
print(f"vs baseline:      {(pred.all_costs[pred.best_model_name] / pred.baseline_cost - 1) * 100:+.1f}%")

print(f"\n--- Model A (Independent Newsvendor) ---")
print(f"Best optimizer:   {best_a.method}")
print(f"Objective:        ${best_a.objective_value:,.2f}")
print(f"Total spend:      ${best_a.total_spend:,.2f}")

print(f"\n--- Model B (Joint Two-Period) ---")
print(f"Objective:        ${model_b.objective_value:,.2f}")
print(f"Y1 orders:        {model_b.q1.sum():.0f} units (spend ${model_b.y1_spend:,.2f})")
print(f"Y2 orders:        {model_b.q2.sum():.0f} units (spend ${model_b.y2_spend:,.2f})")
print(f"Carryover:        {model_b.expected_carryover.sum():.0f} units")

print(f"\n--- Cost of Myopia ---")
print(f"Model A cost:     ${best_a.objective_value:,.2f}")
print(f"Model B cost:     ${model_b.objective_value:,.2f}")
print(f"Savings:          ${cost_of_myopia:,.2f} ({reduction_pct:.1f}%)")

print(f"\n--- Baseline Comparison (2025 Backtest) ---")
print(f"Agent cost:       ${agent_total:,.2f}")
print(f"Baseline cost:    ${baseline_total:,.2f}")
print(f"Savings:          ${savings:,.2f} ({savings_pct:.1f}%)")

print(f"\n--- Shadow Prices ($/$ extra Y1 budget) ---")
print(f"Model A:          ${shadow_a:.4f}")
print(f"Model B:          ${model_b.y1_shadow_price:.4f}")

print(f"\n{'=' * 60}")
print(f"RECOMMENDATION: Adopt newsvendor ordering (CR=0.36) for all")
print(f"categories. Use Model B (joint) for jerseys to exploit the")
print(f"free carryover hedge — saves ${cost_of_myopia:,.0f} per lifecycle.")
print(f"{'=' * 60}")

---

### AI Tool Disclosure

- **Claude (Anthropic)** was used as the LLM backbone of the agent system at three pipeline stages: problem intake, optimizer code exploration, and result explanation.
- **Claude Code** (Anthropic CLI) assisted with code development, debugging, and report formatting.
- All optimization formulations, data analysis, and business interpretations reflect the team's genuine understanding.